In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq

model = ChatGroq(model="gemma2-9b-it", api_key=GROQ_API_KEY)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x718a30f95110>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x718a3103e090>, model_name='gemma2-9b-it')

In [3]:
from langchain_core.messages import HumanMessage

model.invoke([
    HumanMessage(
        content="Hi, my name is James. I am a software engineer and I love building applications with AI. What can you do for me?"
    )
])

AIMessage(content="Hi James, it's nice to meet you! As an AI assistant, I can be a valuable tool for software engineers like yourself who work with AI. Here are some things I can do to help:\n\n**Code Generation and Assistance:**\n\n* **Generate code snippets:** I can help you write code in various programming languages based on your descriptions or examples.\n* **Complete code:** If you're stuck on a piece of code, I can try to complete it for you based on the context.\n* **Suggest improvements:** I can analyze your code and suggest ways to improve its efficiency, readability, or security.\n\n**AI Concept Exploration and Understanding:**\n\n* **Explain AI concepts:** I can provide explanations and definitions of various AI concepts, algorithms, and techniques.\n* **Compare different AI models:** I can help you understand the strengths and weaknesses of different AI models for specific tasks.\n* **Discuss AI ethics and best practices:** I can offer insights into the ethical considerati

In [ ]:
from langchain_core.messages import AIMessage

model.invoke([
    HumanMessage(
        content="Hi, my name is James. I am a software engineer and I love building applications with AI. What can you do for me?"
    ),
    AIMessage(
        content="Hi James, it's nice to meet you! As an AI assistant, I can be a valuable tool for software engineers like yourself who work with AI. Here are some things I can do to help:\n\n**Code Generation and Assistance:**\n\n* **Generate code snippets:** I can help you write code in various programming languages based on your descriptions or examples.\n* **Complete code:** If you're stuck on a piece of code, I can try to complete it for you based on the context.\n* **Suggest improvements:** I can analyze your code and suggest ways to improve its efficiency, readability, or security.\n\n**AI Concept Exploration and Understanding:**\n\n* **Explain AI concepts:** I can provide explanations and definitions of various AI concepts, algorithms, and techniques.\n* **Compare different AI models:** I can help you understand the strengths and weaknesses of different AI models for specific tasks.\n* **Discuss AI ethics and best practices:** I can offer insights into the ethical considerations and best practices surrounding AI development and deployment.\n\n**Data and Research:**\n\n* **Find relevant research papers:** I can search for and summarize academic papers on AI topics of your interest.\n* **Generate data for testing:** I can create synthetic data for testing your AI applications.\n* **Identify potential biases in data:** I can help you analyze your data for potential biases that might affect your AI models.\n\n**Other:**\n\n* **Brainstorm ideas:** I can assist you in brainstorming new ideas for AI applications or features.\n* **Document your code:** I can help you generate documentation for your AI projects.\n\n**To get started, tell me what you're currently working on or what specific AI task you'd like help with.**\n\nI'm excited to see how we can collaborate!\n"
    ),
    HumanMessage(
        content="Hey, what's my name and what do I do?"
    )
])

AIMessage(content="You told me your name is James, and you're a software engineer who loves building applications with AI.  \n\nIs there anything else you'd like to tell me about your work or what you're interested in today? 😊  \n\n", response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 426, 'total_tokens': 479, 'completion_time': 0.096363636, 'prompt_time': 0.008753861, 'queue_time': 0.022139105, 'total_time': 0.105117497}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-14e12fc2-2cdc-4b82-8e53-4882f357a76e-0', usage_metadata={'input_tokens': 426, 'output_tokens': 53, 'total_tokens': 479})

In [7]:
## Message History
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [8]:
config = {"configurable": {"session_id": "chat1"}}

In [11]:
response = with_message_history.invoke(
    [
        HumanMessage(
            content="Hi, my name is James. I am a software engineer"
        ),
    ],
    config=config
)

In [12]:
response.content

"Hi James, that's great!  \n\nI'm Gemma, an AI assistant. What kind of software engineering do you specialize in?  \n\nAre you working on any cool projects right now? 🤔  \n\n"

In [13]:
response = with_message_history.invoke(
    [
        HumanMessage(
            content="What is my name?"
        ),
    ],
    config=config
)

In [14]:
response.content

'Your name is James. 😊  I remember that you told me earlier. \n\nIs there anything else I can help you with, James?\n'

In [15]:
## Change the config --> session_id
config2 = {"configurable": {"session_id": "chat2"}}
response = with_message_history.invoke(
    [
        HumanMessage(
            content="What is my name?"
        ),
    ],
    config=config2
)

In [16]:
response.content

"As an AI, I have no memory of past conversations and no access to personal information about you. So I don't know your name.\n\nIf you'd like to tell me your name, I'd be happy to use it!\n"

In [17]:
response = with_message_history.invoke(
    [
        HumanMessage(
            content="Hi, my name is John?"
        ),
    ],
    config=config2
)
response.content

"Hi John, it's nice to meet you!\n\nIs there anything I can help you with today?\n"

# Prompt Templates

In [19]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all questions to the best of your ability."),
        MessagesPlaceholder(variable_name="messages")
    ]
)
chain = prompt | model

In [20]:
chain.invoke({
    "messages": [
        HumanMessage(
            content="Hi, my name is James. I am a software engineer."
        ),
    ], 
})

AIMessage(content="Hello James, it's nice to meet you! \n\nI'm glad to hear you're a software engineer. What can I do for you today? Do you have any questions I can help answer, or tasks I can assist with? \n\nI'm here to help in any way I can!\n", response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 38, 'total_tokens': 106, 'completion_time': 0.123636364, 'prompt_time': 0.001555829, 'queue_time': 0.08799730700000001, 'total_time': 0.125192193}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-4a162b6d-5896-43ce-9c40-ce46ee5df3e2-0', usage_metadata={'input_tokens': 38, 'output_tokens': 68, 'total_tokens': 106})

In [21]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [22]:
config3 = {"configurable": {"session_id": "chat3"}}

response = with_message_history.invoke(
    [
        HumanMessage(
            content="Hi, my name is James. I am a software engineer."
        ),
    ],
    config=config3
)

response.content

"Hello James! Nice to meet you. I'm glad you're here.\n\nHow can I help you today? Do you have any questions about software engineering, need help with a coding problem, or just want to chat?  I'm here to assist in any way I can.\n"

In [23]:
generic_template = "Translate the following into {language}"

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", generic_template),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | model

In [24]:
response = chain.invoke({
    "messages": [
        HumanMessage(
            content="Hi, my name is James."
        ),
    ],
    "language": "Spanish"
})
response.content

'Hola, mi nombre es James. \n'

Let's now wrap this in more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [25]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history, input_messages_key="messages")

In [26]:
config4 = {"configurable": {"session_id": "chat4"}}

response = with_message_history.invoke({
    "messages": [
        HumanMessage(
            content="Hi, my name is James."
        ),
    ],
    "language": "Spanish"
}, config=config4)

response.content

'Hola, me llamo James. \n'

In [27]:
response = with_message_history.invoke({
    "messages": [
        HumanMessage(
            content="What is my name?"
        ),
    ],
    "language": "Spanish"
}, config=config4)

response.content

'Your name is James, because you told me so! 😊  \n\nIs there anything else I can help you with?\n'

# Managing Conversation History

Messages could overflow context window if not managed, so you need to limit the size of the messages you are passing.

'trim_messages' helper reduces the messages we're sending in the model. Specify tokens to keep along with parameters like if we always want to keep the system message and whether to allow partial messages.

In [32]:
from langchain_core.messages import SystemMessage, trim_messages
trimmer = trim_messages(
    max_tokens=70,
    strategy='last',
    token_counter=model,
    include_system=True,
    allow_partial=True,
    start_on="human"
)

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Hi, I'm Bob"),
    AIMessage(content="Hi!"),
    HumanMessage(content="I like chocolate ice cream."),
    AIMessage(content="That's great!"),
    HumanMessage(content="What is 2+2?"),
    AIMessage(content="It's 4."),
    HumanMessage(content="Thanks!"),
    AIMessage(content="You're welcome!"),
    HumanMessage(content="Having fun?"),
    AIMessage(content="Yes, I am!"),
]

trimmer.invoke(messages)

[SystemMessage(content='You are a helpful assistant.'),
 HumanMessage(content="Hi, I'm Bob"),
 AIMessage(content='Hi!'),
 HumanMessage(content='I like chocolate ice cream.'),
 AIMessage(content="That's great!"),
 HumanMessage(content='What is 2+2?'),
 AIMessage(content="It's 4."),
 HumanMessage(content='Thanks!'),
 AIMessage(content="You're welcome!"),
 HumanMessage(content='Having fun?'),
 AIMessage(content='Yes, I am!')]

In [33]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(
        messages=itemgetter("messages") | trimmer)
    | prompt
    | model
)

response = chain.invoke({
    "messages": messages + [
        HumanMessage(
            content="What ice cream do I like?"
        ),
    ],
    "language": "English"
})

response.content

'You like chocolate ice cream!  🍫 😋 \n\n\n'

In [34]:
response = chain.invoke({
    "messages": messages + [
        HumanMessage(
            content="What math problem did I ask?"
        ),
    ],
    "language": "English"
})

response.content

'You asked "What is 2+2?" \n\n\n\nLet me know if you\'d like to try another one!\n'

In [35]:
# Let's wrap this in the message history

with_message_history = RunnableWithMessageHistory(chain, get_session_history, input_messages_key="messages")

config5 = {"configurable": {"session_id": "chat5"}}

In [36]:
response = with_message_history.invoke({
    "messages": messages + [
        HumanMessage(
            content="What is my name?"
        ),
    ],
    "language": "English"
}, config=config5)

response.content

"As an AI, I don't have access to past conversations or personal information about you. So I don't know your name. 😉\n\nWould you like to tell me?\n"